# Assignment 8 — NLP Using Python II

**Jiya Mahajan | Roll No. 1024170296 | Batch 3Q31 / 3Q3A**


In [ ]:
import nltk
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(resource, quiet=True)


## Q1


In [ ]:
paragraph="Artificial intelligence fascinates me because it helps people solve practical problems. I enjoy learning how machines recognize patterns in images and language. Responsible technology should protect privacy and make decisions easier to understand. Small experiments with data can reveal useful insights that are difficult to notice by intuition alone. I hope to build thoughtful tools that support education, healthcare, and creativity."
import re, collections
text_lower=paragraph.lower(); clean=re.sub(r"[^\w\s]","",text_lower)
print("Clean text:",clean)
print("Sentences:",re.split(r"(?<=[.!?])\s+",paragraph))
print("split():",clean.split())
from nltk.tokenize import word_tokenize
print("word_tokenize():",word_tokenize(text_lower))
from nltk.corpus import stopwords
stop_words=set(stopwords.words("english"))
tokens=[w for w in re.findall(r"\b[a-z]+\b",clean) if w not in stop_words]
print("Without stopwords:",tokens)
print("Frequency:",collections.Counter(tokens))

`split()` separates on whitespace and can leave punctuation attached. `word_tokenize()` recognizes punctuation as separate tokens and applies language-aware tokenization rules.

## Q2


In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
words=re.findall(r"[A-Za-z]+",paragraph.lower()); filtered=[w for w in words if w not in stop_words]
porter=PorterStemmer(); lemma=WordNetLemmatizer()
print("Alphabetic words:",words); print("Filtered:",filtered)
print("Porter stems:",[porter.stem(w) for w in filtered])
print("Lemmas:",[lemma.lemmatize(w) for w in filtered])

Stemming is fast and useful for rough search matching but can produce non-words. Lemmatization uses vocabulary and grammar to return dictionary forms, useful when semantic readability matters.

## Q3


In [ ]:
texts=["AI assistants help people search documents quickly.","Solar panels convert sunlight into clean electricity.","Machine learning systems discover useful patterns from data."]
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
count_vectorizer=CountVectorizer(); bow=count_vectorizer.fit_transform(texts)
print("Vocabulary:",count_vectorizer.get_feature_names_out()); print("Bag of Words:\n",bow.toarray())
tfidf_vectorizer=TfidfVectorizer(stop_words="english"); tfidf=tfidf_vectorizer.fit_transform(texts); features=tfidf_vectorizer.get_feature_names_out()
for i,row in enumerate(tfidf.toarray()):
    top=row.argsort()[::-1][:3]
    print(f"Text {i+1} top keywords:",[(features[j],round(row[j],3)) for j in top if row[j]>0])

High TF-IDF terms are informative within a text and uncommon across the collection, distinguishing the three topics.

## Q4


In [ ]:
technology_a="Artificial intelligence learns patterns from examples. It can classify images, summarize documents, and assist with decisions. Training data and evaluation help improve an AI model. Careful design helps reduce bias and protect user privacy."
technology_b="Blockchain stores transactions in linked records. A distributed network verifies updates without relying on one central database. Cryptographic methods help participants audit the shared ledger. Smart contracts can automate agreements between parties."
import re
def preprocess(text): return re.findall(r"\b[a-z]+\b",text.lower())
a=preprocess(technology_a); b=preprocess(technology_b); set_a,set_b=set(a),set(b)
jaccard=len(set_a & set_b)/len(set_a | set_b)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
vectors=TfidfVectorizer().fit_transform([technology_a,technology_b]); cosine=float(cosine_similarity(vectors[0],vectors[1])[0,0])
print("Jaccard:",jaccard); print("Cosine:",cosine)

Jaccard compares word presence and ignores repetition. TF-IDF cosine weights words by informativeness and accommodates documents of different lengths. Both should show low overlap here.

## Q5


In [ ]:
reviews=["The headphones sound clear and the battery lasts all day.","The delivery was late and the package arrived damaged.","The notebook is fine and works as described."]
from textblob import TextBlob
positive=[]
for review in reviews:
    sentiment=TextBlob(review).sentiment
    label="Positive" if sentiment.polarity>0.1 else "Negative" if sentiment.polarity < -0.1 else "Neutral"
    print(review,"| polarity:",round(sentiment.polarity,3),"| subjectivity:",round(sentiment.subjectivity,3),"|",label)
    if label=="Positive": positive.append(review)
from wordcloud import WordCloud
import matplotlib.pyplot as plt
plt.imshow(WordCloud(width=800,height=400,background_color="white").generate(" ".join(positive)))
plt.axis("off"); plt.show()

## Q6


In [ ]:
training_text="curious students explore artificial intelligence and build helpful tools. data teaches a model to recognize patterns and make useful predictions. creative teams test their ideas, learn from results, and improve their designs. responsible technology supports people and makes complex work easier."
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
tokenizer=Tokenizer(); tokenizer.fit_on_texts([training_text]); sequences=[]
for line in training_text.split("."):
    ids=tokenizer.texts_to_sequences([line])[0]
    for i in range(1,len(ids)): sequences.append(ids[:i+1])
max_len=max(map(len,sequences)); sequences=pad_sequences(sequences,maxlen=max_len,padding="pre")
X_train=sequences[:,:-1]; y_train=sequences[:,-1]
model=Sequential([Embedding(len(tokenizer.word_index)+1,32,input_length=max_len-1),LSTM(64),Dense(len(tokenizer.word_index)+1,activation="softmax")])
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy"); model.fit(X_train,y_train,epochs=80,verbose=0)
def generate(seed,words=8):
    output=seed
    for _ in range(words):
        seq=pad_sequences(tokenizer.texts_to_sequences([output]),maxlen=max_len-1,padding="pre")
        next_id=int(model.predict(seq,verbose=0).argmax()); next_word=tokenizer.index_word.get(next_id,"")
        if not next_word: break
        output += " " + next_word
    return output
print(generate("curious students")); print(generate("responsible technology")); print(generate("data teaches"))